# Tutorial: Platforma Ewaluacji Systemów Śledzenia Partytury

Ten notebook przeprowadzi Cię krok po kroku przez:
1. Przygotowanie danych
2. Testowanie modelu DTW (baseline)
3. Porównanie modeli
4. Analizę wyników

## Setup

In [ ]:
# Instalacja zależności (odkomentuj jeśli potrzebne)
# !pip install -r requirements.txt

import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import librosa
import librosa.display

# Dodaj projekt do path
sys.path.append('..')

from models.otw_model import DTWModel, OnlineTimeWarping
from evaluation.evaluator import Evaluator
from utils.audio_processing import AudioProcessor
from utils.midi_processing import MIDIProcessor

print("✓ Imports ready!")

## 1. Przygotowanie Danych

Potrzebujesz:
- Plik audio (.wav, .mp3)
- Plik MIDI (.mid) jako referencja

Możesz użyć MAESTRO dataset lub własnych plików.

In [ ]:
# Ustaw ścieżki do swoich plików
AUDIO_PATH = "path/to/your/audio.wav"  # ZMIEŃ TO!
MIDI_PATH = "path/to/your/reference.mid"  # ZMIEŃ TO!

# Sprawdź czy pliki istnieją
import os
if os.path.exists(AUDIO_PATH):
    print(f"✓ Audio found: {AUDIO_PATH}")
else:
    print(f"✗ Audio NOT found: {AUDIO_PATH}")
    print("  Please set correct path!")

if os.path.exists(MIDI_PATH):
    print(f"✓ MIDI found: {MIDI_PATH}")
else:
    print(f"✗ MIDI NOT found: {MIDI_PATH}")
    print("  Please set correct path!")

### Wizualizacja danych

In [ ]:
# Wczytaj i wizualizuj audio
processor = AudioProcessor()
audio, sr = processor.load_audio(AUDIO_PATH)

print(f"Audio duration: {len(audio)/sr:.2f} seconds")
print(f"Sample rate: {sr} Hz")

# Spektrogram
spec = processor.compute_spectrogram(audio, use_mel=True)

plt.figure(figsize=(14, 5))
librosa.display.specshow(spec, sr=sr, x_axis='time', y_axis='mel')
plt.colorbar(format='%+2.0f dB')
plt.title('Mel Spectrogram')
plt.tight_layout()
plt.show()

In [ ]:
# Wczytaj i wizualizuj MIDI
midi_processor = MIDIProcessor()
midi = midi_processor.load_midi(MIDI_PATH)

print(f"MIDI duration: {midi_processor.get_duration(midi):.2f} seconds")
print(f"Number of instruments: {len(midi.instruments)}")

# Piano roll
piano_roll = midi_processor.midi_to_piano_roll(midi)

plt.figure(figsize=(14, 5))
plt.imshow(piano_roll, aspect='auto', origin='lower', cmap='hot', interpolation='nearest')
plt.colorbar()
plt.ylabel('MIDI Note Number')
plt.xlabel('Time (frames)')
plt.title('Piano Roll')
plt.tight_layout()
plt.show()

## 2. Test Pojedynczego Modelu (DTW)

Zacznijmy od prostego modelu DTW jako baseline.

In [ ]:
# Stwórz model DTW
dtw_model = DTWModel(window_size=100, feature_type='chroma')

print(f"Model: {dtw_model.name}")
print(f"Requires training: {dtw_model.requires_training()}")
print(f"Window size: {dtw_model.window_size} frames")

In [ ]:
# Ewaluuj model
evaluator = Evaluator(tolerance_seconds=0.5)

print("Starting evaluation...")
metrics = evaluator.evaluate_single_model(
    model=dtw_model,
    audio_path=AUDIO_PATH,
    reference_path=MIDI_PATH,
    verbose=True
)

print("\n" + "="*60)
print("RESULTS:")
print(metrics)

## 3. Porównanie Modeli DTW vs OTW

In [ ]:
# Stwórz oba modele
dtw = DTWModel(window_size=100)
otw = OnlineTimeWarping(window_size=100, search_margin=30)

# Porównaj
results = evaluator.compare_all_models(
    models=[dtw, otw],
    audio_path=AUDIO_PATH,
    reference_path=MIDI_PATH,
    save_results=True
)

## 4. Wizualizacja Wyników

In [ ]:
# Wykres porównawczy
import pandas as pd

# Przygotuj dane
data = {
    'Model': list(results.keys()),
    'Accuracy': [m.frame_accuracy for m in results.values()],
    'Mean Error (s)': [m.mean_error for m in results.values()],
    'Latency (ms)': [m.mean_latency for m in results.values()]
}

df = pd.DataFrame(data)

# Wykresy
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Accuracy
axes[0].bar(df['Model'], df['Accuracy'])
axes[0].set_ylabel('Frame Accuracy')
axes[0].set_title('Frame Accuracy')
axes[0].set_ylim([0, 1])

# Error
axes[1].bar(df['Model'], df['Mean Error (s)'])
axes[1].set_ylabel('Mean Error (seconds)')
axes[1].set_title('Mean Error')

# Latency
axes[2].bar(df['Model'], df['Latency (ms)'])
axes[2].set_ylabel('Latency (ms)')
axes[2].set_title('Mean Latency')

plt.tight_layout()
plt.show()

# Pokaż tabelę
print("\nDetailed Results:")
print(df.to_string(index=False))

## 5. Test Tempo Robustness

In [ ]:
# Testuj różne tempa
tempo_ratios = [0.8, 0.9, 1.0, 1.1, 1.2]

tempo_results = evaluator.evaluate_tempo_robustness(
    model=dtw,
    audio_path=AUDIO_PATH,
    reference_path=MIDI_PATH,
    tempo_ratios=tempo_ratios,
    verbose=False
)

# Wizualizacja
tempos = list(tempo_results.keys())
accuracies = [m.frame_accuracy for m in tempo_results.values()]
errors = [m.mean_error for m in tempo_results.values()]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot([t*100 for t in tempos], accuracies, 'o-', linewidth=2, markersize=8)
axes[0].set_xlabel('Tempo (%)')
axes[0].set_ylabel('Frame Accuracy')
axes[0].set_title('Accuracy vs Tempo')
axes[0].grid(True, alpha=0.3)

axes[1].plot([t*100 for t in tempos], errors, 'o-', linewidth=2, markersize=8, color='red')
axes[1].set_xlabel('Tempo (%)')
axes[1].set_ylabel('Mean Error (s)')
axes[1].set_title('Error vs Tempo')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Następne Kroki

1. **Zbierz więcej danych**: Użyj MAESTRO dataset do szerszej ewaluacji
2. **Zaimplementuj trening CNN**: Wypełnij metodę `train()` w `cnn_model.py`
3. **Dodaj więcej metryk**: Rozszerz `metrics.py` o własne metryki
4. **Testuj przypadki brzegowe**: Pomyłki, ozdobniki, zmienność tempa

Powodzenia w pracy magisterskiej! 🎵